# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Matplotlib is building the font cache; this may take a moment.


## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [ ]:
# Load the dataset
df = pd.read_csv("data/AviationData.csv", encoding="cp1252")



C:\Users\Administrator\AppData\Local\Temp\ipykernel_24624\3216887979.py:2: DtypeWarning: Columns (0: Latitude, 1: Longitude, 2: Broad.phase.of.flight) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/AviationData.csv", encoding="cp1252")


In [8]:
df.head()
df.shape
df.info()
df.isna().sum()
df.dtypes
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [13]:
cols = [
    'Event.Date', 'Aircraft.Category', 'Amateur.Built', 'Make', 'Model',
    'Aircraft.damage', 'Purpose.of.flight', 'Schedule', 'Weather.Condition',
    'Broad.phase.of.flight', 'Number.of.Engines',
    'Total.Fatal.Injuries', 'Total.Serious.Injuries',
    'Total.Minor.Injuries', 'Total.Uninjured'
]

df[cols].head()

,Event.Date,Aircraft.Category,Amateur.Built,Make,Model,Aircraft.damage,Purpose.of.flight,Schedule,Weather.Condition,Broad.phase.of.flight,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
3600,1983-01-01,NaN,No,Piccard,AX-6,NaN,Personal,NaN,VMC,Landing,NaN,0.0,1.0,0.0,1.0
3601,1983-01-01,NaN,No,Cessna,182P,Substantial,Personal,NaN,VMC,Approach,1.0,0.0,0.0,1.0,3.0
3602,1983-01-01,NaN,No,Cessna,182RG,Substantial,Personal,NaN,VMC,Landing,1.0,0.0,0.0,0.0,2.0
3603,1983-01-01,NaN,No,Cessna,182P,Substantial,Personal,NaN,VMC,Takeoff,1.0,0.0,0.0,0.0,1.0
3604,1983-01-01,NaN,No,Piper,PA-28R-200,Substantial,Personal,NaN,VMC,Approach,1.0,0.0,0.0,2.0,0.0


In [14]:
df[cols].isna().sum()
df[cols].dtypes

Event.Date                datetime64[us]
Aircraft.Category                    str
Amateur.Built                        str
Make                                 str
Model                                str
Aircraft.damage                      str
Purpose.of.flight                    str
Schedule                             str
Weather.Condition                    str
Broad.phase.of.flight                str
Number.of.Engines                float64
Total.Fatal.Injuries             float64
Total.Serious.Injuries           float64
Total.Minor.Injuries             float64
Total.Uninjured                  float64
dtype: object

In [11]:
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')
df = df.dropna(subset=['Event.Date'])
df = df[df['Event.Date'].dt.year >= 1983]

df['Amateur.Built'] = df['Amateur.Built'].str.strip().str.title()
df = df[df['Amateur.Built'] != 'Yes']

In [12]:
df['Aircraft.damage'] = df['Aircraft.damage'].str.strip().str.title()

In [15]:

df_clean = df.copy()


df_clean['Event.Date'] = pd.to_datetime(df_clean['Event.Date'], errors='coerce')


text_cols = [
    'Aircraft.Category', 'Amateur.Built', 'Aircraft.damage', 'Engine.Type',
    'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Weather.Condition',
    'Broad.phase.of.flight', 'Investigation.Type'
]

for col in text_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.strip().str.title()
        df_clean[col] = df_clean[col].replace({'Nan': np.nan, 'None': np.nan, '': np.nan})


injury_cols = [
    'Total.Fatal.Injuries', 'Total.Serious.Injuries',
    'Total.Minor.Injuries', 'Total.Uninjured'
]

for col in injury_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')



In [16]:
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')
df = df.dropna(subset=['Event.Date'])
df = df[df['Event.Date'].dt.year >= 1983]
df['Amateur.Built'] = df['Amateur.Built'].astype(str).str.strip().str.title()
df = df[df['Amateur.Built'] != 'Yes']

In [17]:
for c in ['Aircraft.damage', 'Purpose.of.flight', 'Weather.Condition', 'Broad.phase.of.flight']:
    df[c] = df[c].astype(str).str.strip().str.title()

In [18]:
df.shape
df[cols].isna().sum()
df.info()

<class 'pandas.DataFrame'>
Index: 77061 entries, 3600 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                77061 non-null  str           
 1   Investigation.Type      77061 non-null  str           
 2   Accident.Number         77061 non-null  str           
 3   Event.Date              77061 non-null  datetime64[us]
 4   Location                77010 non-null  str           
 5   Country                 76851 non-null  str           
 6   Latitude                30198 non-null  object        
 7   Longitude               30192 non-null  object        
 8   Airport.Code            43385 non-null  str           
 9   Airport.Name            45295 non-null  str           
 10  Injury.Severity         76062 non-null  str           
 11  Aircraft.damage         73951 non-null  str           
 12  Aircraft.Category       25423 non-null  str           
 13 

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [20]:

injury_cols = [
    'Total.Fatal.Injuries', 'Total.Serious.Injuries',
    'Total.Minor.Injuries', 'Total.Uninjured'
]

for col in injury_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')


df_clean[injury_cols] = df_clean[injury_cols].fillna(0)


df_clean['Estimated.Passengers'] = (
    df_clean['Total.Fatal.Injuries']
    + df_clean['Total.Serious.Injuries']
    + df_clean['Total.Minor.Injuries']
    + df_clean['Total.Uninjured']
)


df_clean['Estimated.Passengers'] = df_clean['Estimated.Passengers'].replace(0, np.nan)


df_clean['Fatal_Serious_Rate'] = (
    (df_clean['Total.Fatal.Injuries'] + df_clean['Total.Serious.Injuries'])
    / df_clean['Estimated.Passengers']
)

df_clean['Fatal_Serious_Rate'] = df_clean['Fatal_Serious_Rate'].fillna(0)

In [21]:
df_clean['Destroyed'] = df_clean['Aircraft.damage'].str.contains('Destroyed', case=False, na=False).astype(int)

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [22]:

df_clean['Aircraft.damage'] = (
    df_clean['Aircraft.damage']
    .astype(str)
    .str.strip()
    .str.title()
    .replace({'Nan': np.nan, 'None': np.nan, '': np.nan})
)


df_clean['Destroyed'] = df_clean['Aircraft.damage'].str.contains(
    'Destroyed', case=False, na=False
).astype(int)


df_clean['Aircraft.damage'].value_counts(dropna=False)
df_clean['Destroyed'].value_counts(dropna=False)

Destroyed
0    61558
1    15503
Name: count, dtype: int64

In [23]:
df_clean[['Aircraft.damage', 'Destroyed']].head(10)

,Aircraft.damage,Destroyed
3600,NaN,0
3601,Substantial,0
3602,Substantial,0
3603,Substantial,0
3604,Substantial,0
3605,Substantial,0
3606,NaN,0
3607,Destroyed,1
3608,Destroyed,1
3609,Destroyed,1


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [24]:

df_clean['Make'] = (
    df_clean['Make']
    .astype(str)
    .str.strip()
    .str.title()
    .replace({'Nan': np.nan, 'None': np.nan, '': np.nan})
)


df_clean = df_clean.dropna(subset=['Make'])


make_counts = df_clean['Make'].value_counts()
make_counts.head(20)

Make
Cessna               25758
Piper                14107
Beech                 5098
Boeing                2682
Bell                  2595
Mooney                1271
Robinson              1198
Grumman               1064
Bellanca               978
Hughes                 877
Schweizer              754
Air Tractor            673
Aeronca                607
Mcdonnell Douglas      599
Maule                  571
Champion               504
Stinson                421
Aero Commander         411
De Havilland           405
Luscombe               390
Name: count, dtype: int64

In [25]:
threshold = 50
valid_makes = make_counts[make_counts >= threshold].index
df_clean = df_clean[df_clean['Make'].isin(valid_makes)]

In [26]:
df_clean['Make'].value_counts().head(10)
df_clean.shape

(70562, 34)

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [27]:

df_clean['Model'] = (
    df_clean['Model']
    .astype(str)
    .str.strip()
    .str.title()
    .replace({'Nan': np.nan, 'None': np.nan, '': np.nan})
)


df_clean = df_clean.dropna(subset=['Model'])


make_model_counts = (
    df_clean.groupby(['Make', 'Model'])
    .size()
    .reset_index(name='Count')
    .sort_values('Count', ascending=False)
)

make_model_counts.head(20)

,Make,Model,Count
2511,Cessna,152,2229
2534,Cessna,172,1650
2585,Cessna,172N,1093
5380,Piper,Pa-28-140,865
2583,Cessna,172M,759
2484,Cessna,150,752
2588,Cessna,172P,665
2640,Cessna,182,617
2617,Cessna,180,595
2510,Cessna,150M,551


In [28]:
model_make_nunique = df_clean.groupby('Model')['Make'].nunique().sort_values(ascending=False)
model_make_nunique.head(20)

Model
500       7
G-164A    7
G-164B    6
Aa-5      5
Aa-5B     5
G-164     5
G-164C    5
690C      4
Aa-5A     4
300       4
700       4
Aa-1C     4
8Kcab     4
7Ec       4
7Eca      4
400       4
100       4
Aa-1B     4
690A      4
269C      4
Name: Make, dtype: int64

In [30]:
df_clean['Plane.Type'] = df_clean['Make'].astype(str).str.strip() + " " + df_clean['Model'].astype(str).str.strip()

In [31]:
df_clean[['Make', 'Model', 'Plane.Type']].head()
df_clean['Plane.Type'].nunique()

6558

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [32]:
for col in ['Engine.Type', 'Weather.Condition', 'Number.of.Engines', 'Purpose.of.flight', 'Broad.phase.of.flight']:
    print(f"\n{col}")
    print(df_clean[col].value_counts(dropna=False).head(10))


Engine.Type
Engine.Type
Reciprocating      55014
NaN                 5491
Turbo Shaft         2989
Turbo Prop          2796
Turbo Fan           2178
Unknown             1514
Turbo Jet            543
Geared Turbofan       12
Lr                     1
Unk                    1
Name: count, dtype: int64

Weather.Condition
Weather.Condition
Vmc    60586
Imc     5269
NaN     3751
Unk      933
Name: count, dtype: int64

Number.of.Engines
Number.of.Engines
1.0    54458
2.0     9611
NaN     4896
0.0      741
3.0      431
4.0      401
8.0        1
Name: count, dtype: int64

Purpose.of.flight
Purpose.of.flight
Personal              36923
Instructional          9393
Unknown                5507
NaN                    5389
Aerial Application     4134
Business               3432
Positioning            1446
Other Work Use         1061
Aerial Observation      708
Public Aircraft         643
Name: count, dtype: int64

Broad.phase.of.flight
Broad.phase.of.flight
NaN            20478
Landing        13011


In [33]:

df_clean2 = df_clean.copy()


df_clean2['Engine.Type'] = (
    df_clean2['Engine.Type']
    .astype(str)
    .str.strip()
    .str.title()
    .replace({'Nan': np.nan, 'None': np.nan, '': np.nan})
)


df_clean2['Weather.Condition'] = (
    df_clean2['Weather.Condition']
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({'NAN': np.nan, 'NONE': np.nan, '': np.nan})
)


df_clean2['Number.of.Engines'] = pd.to_numeric(df_clean2['Number.of.Engines'], errors='coerce')


df_clean2['Purpose.of.flight'] = (
    df_clean2['Purpose.of.flight']
    .astype(str)
    .str.strip()
    .str.title()
    .replace({'Nan': np.nan, 'None': np.nan, '': np.nan})
)


df_clean2['Broad.phase.of.flight'] = (
    df_clean2['Broad.phase.of.flight']
    .astype(str)
    .str.strip()
    .str.title()
    .replace({'Nan': np.nan, 'None': np.nan, '': np.nan})
)

In [34]:
for col in ['Engine.Type', 'Weather.Condition', 'Number.of.Engines', 'Purpose.of.flight', 'Broad.phase.of.flight']:
    print(f"\n{col}")
    print(df_clean[col].value_counts(dropna=False).head(10))


Engine.Type
Engine.Type
Reciprocating      55014
NaN                 5491
Turbo Shaft         2989
Turbo Prop          2796
Turbo Fan           2178
Unknown             1514
Turbo Jet            543
Geared Turbofan       12
Lr                     1
Unk                    1
Name: count, dtype: int64

Weather.Condition
Weather.Condition
Vmc    60586
Imc     5269
NaN     3751
Unk      933
Name: count, dtype: int64

Number.of.Engines
Number.of.Engines
1.0    54458
2.0     9611
NaN     4896
0.0      741
3.0      431
4.0      401
8.0        1
Name: count, dtype: int64

Purpose.of.flight
Purpose.of.flight
Personal              36923
Instructional          9393
Unknown                5507
NaN                    5389
Aerial Application     4134
Business               3432
Positioning            1446
Other Work Use         1061
Aerial Observation      708
Public Aircraft         643
Name: count, dtype: int64

Broad.phase.of.flight
Broad.phase.of.flight
NaN            20478
Landing        13011


In [35]:
df_clean2[['Engine.Type', 'Weather.Condition', 'Number.of.Engines', 'Purpose.of.flight', 'Broad.phase.of.flight']].head()
df_clean2.info()

<class 'pandas.DataFrame'>
Index: 70539 entries, 3601 to 88888
Data columns (total 35 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                70539 non-null  str           
 1   Investigation.Type      70539 non-null  str           
 2   Accident.Number         70539 non-null  str           
 3   Event.Date              70539 non-null  datetime64[us]
 4   Location                70491 non-null  str           
 5   Country                 70345 non-null  str           
 6   Latitude                25980 non-null  object        
 7   Longitude               25975 non-null  object        
 8   Airport.Code            39815 non-null  str           
 9   Airport.Name            41666 non-null  str           
 10  Injury.Severity         69659 non-null  str           
 11  Aircraft.damage         67843 non-null  str           
 12  Aircraft.Category       21548 non-null  str           
 13 

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [36]:
missing_counts = df_clean2.isna().sum().sort_values(ascending=False)
missing_pct = (df_clean2.isna().mean() * 100).sort_values(ascending=False)

missing_summary = pd.DataFrame({
    'missing_count': missing_counts,
    'missing_pct': missing_pct
})

missing_summary.head(20)

,missing_count,missing_pct
Schedule,59664,84.582997
Air.carrier,57277,81.199053
FAR.Description,49184,69.725967
Aircraft.Category,48991,69.452360
Longitude,44564,63.176399
Latitude,44559,63.169311
Airport.Code,30724,43.556047
Airport.Name,28873,40.931967
Broad.phase.of.flight,20478,29.030749
Publication.Date,12198,17.292562


In [37]:
threshold = 50
cols_to_drop = missing_summary[missing_summary['missing_pct'] > threshold].index.tolist()

df_clean3 = df_clean2.drop(columns=cols_to_drop)

cols_to_drop

['Schedule',
 'Air.carrier',
 'FAR.Description',
 'Aircraft.Category',
 'Longitude',
 'Latitude']

In [38]:
df_clean3.shape
df_clean3.info()

<class 'pandas.DataFrame'>
Index: 70539 entries, 3601 to 88888
Data columns (total 29 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                70539 non-null  str           
 1   Investigation.Type      70539 non-null  str           
 2   Accident.Number         70539 non-null  str           
 3   Event.Date              70539 non-null  datetime64[us]
 4   Location                70491 non-null  str           
 5   Country                 70345 non-null  str           
 6   Airport.Code            39815 non-null  str           
 7   Airport.Name            41666 non-null  str           
 8   Injury.Severity         69659 non-null  str           
 9   Aircraft.damage         67843 non-null  str           
 10  Registration.Number     69385 non-null  str           
 11  Make                    70539 non-null  str           
 12  Model                   70539 non-null  str           
 13 

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [39]:
df_clean3.to_csv("data/AviationData_cleaned.csv", index=False)